# Phase 2: Advanced Feature Engineering (v2)

In this version, we're adding more sophisticated features to boost our ROC-AUC towards 0.88:
- Customer Lifetime Value (CLV) Estimation
- Engagement Scores
- Tenure-Service Interactions
- MonthlySpend-Contract interactions


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


In [3]:
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['EngagementScore'] = df[services].apply(lambda x: (x == 'Yes').sum(), axis=1)
df['CLV'] = df['MonthlyCharges'] * df['tenure']
df['AvgCharge'] = df['TotalCharges'] / (df['tenure'] + 1)
df['LoyalEngaged'] = df['EngagementScore'] * df['tenure']
contract_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df['ContractValue'] = df['Contract'].map(contract_map) * df['MonthlyCharges']


In [4]:
target = 'Churn'
drop_cols = ['customerID']
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col not in drop_cols + [target]]
df_final = pd.get_dummies(df.drop(columns=drop_cols), columns=categorical_cols, drop_first=True)


In [5]:
X = df_final.drop(columns=[target])
y = df_final[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)


In [6]:
np.save('../data/X_train_adv.npy', X_train_scaled.values)
np.save('../data/X_test_adv.npy', X_test_scaled.values)
np.save('../data/y_train_adv.npy', y_train.values)
np.save('../data/y_test_adv.npy', y_test.values)
with open('../data/feature_names_adv.pkl', 'wb') as f: pickle.dump(X_train.columns.tolist(), f)
